In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import operator

In [2]:
ticker_file = pd.read_csv('Tickers_Example.csv', header=None)
ticker_file.rename(columns={0: 'NAME'}, inplace=True)

In [3]:
def clean_data(tickers):
    filtered_stocks = pd.DataFrame()
    start_date = "2023-10-01"
    end_date = "2024-09-30"
    for ticker in tickers['NAME']:
        try:
            stock = yf.Ticker(ticker)
            info = stock.fast_info
            if info['currency'] not in ["USD", "CAD"]: # Enusring stock is listed, traded in CAD or USD
                continue
    
            hist = stock.history(start=start_date, end=end_date, interval="1d")
            hist['Month'] = hist.index.to_period('M')
            monthly_data = hist.groupby('Month').filter(lambda x: len(x) >= 18)

            avg_monthly_volume = monthly_data.groupby('Month')['Volume'].mean().mean()
            if avg_monthly_volume >= 100000:
                filtered_stocks = pd.concat([filtered_stocks, pd.DataFrame({"Ticker": [ticker]})])
                
        except Exception as e:
            continue
    
    return filtered_stocks.reset_index(drop=True)

In [4]:
ticker_file = clean_data(ticker_file)

/var/folders/0n/r4s34nf565330b8vj1m60v3m0000gp/T/ipykernel_476/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/0n/r4s34nf565330b8vj1m60v3m0000gp/T/ipykernel_476/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/0n/r4s34nf565330b8vj1m60v3m0000gp/T/ipykernel_476/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
/var/folders/0n/r4s34nf565330b8vj1m60v3m0000gp/T/ipykernel_476/3848309862.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hist['Month'] = hist.index.to_period('M')
$AGN: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
/var/folders/0n/r4

In [5]:
def getBeta(cur_ticker):
    SP_ticker = '^GSPC'

    cur_stock = yf.Ticker(cur_ticker)
    SP_index = yf.Ticker(SP_ticker)

    # Will edit these later
    start_date = '2022-11-14'
    end_date = '2024-11-14'

    cur_stock_hist = cur_stock.history(start=start_date, end=end_date)
    SP_index_hist = SP_index.history(start=start_date, end=end_date)

    prices = pd.DataFrame()
    prices[cur_ticker] = pd.DataFrame(cur_stock_hist['Close'])
    prices['S&P Index'] = SP_index_hist['Close']

    daily_returns = prices.pct_change(fill_method=None).dropna()
    daily_returns.drop(index=daily_returns.index[0], inplace=True)

    SP_var = daily_returns['S&P Index'].var()
    SP_beta = daily_returns.cov() / SP_var

    return SP_beta.iat[0,1]

In [6]:
# Gets average of daily growth
def getGrowth(cur_ticker):
    # Fetch data for current stock
    cur_stock = yf.Ticker(cur_ticker)
    start_date = '2022-11-14'
    end_date = '2024-11-14'
    cur_stock_hist = cur_stock.history(start=start_date, end=end_date)

    prices = pd.DataFrame()
    prices[cur_ticker] = cur_stock_hist['Close']
    
    daily_returns = prices.pct_change(fill_method=None).dropna()
    SP_growth = daily_returns.mean()
    
    return SP_growth[cur_ticker] * 100

In [7]:
def getVolatility(cur_ticker):
    start_date = '2022-11-14'
    end_date = '2024-11-14'
    stock_data = yf.Ticker(cur_ticker).history(start=start_date, end=end_date)
    stock_data['Daily Return'] = stock_data['Close'].pct_change(fill_method=None).dropna()
    
    volatility = stock_data['Daily Return'].std()
    return volatility

In [13]:
from operator import index

# Initialize data structures
stock_data = []
beta_data = []
volatility_data = []
growth_data = []

# Iterate through the tickers and gather data
for i in range(len(ticker_file)):
    cur_ticker = ticker_file['Ticker'].iloc[i]

    # Replace these functions with your actual implementations
    beta = getBeta(cur_ticker)
    beta_data.append({"name": cur_ticker, "beta": beta})
    growth = getGrowth(cur_ticker)
    growth_data.append({"name": cur_ticker, "growth": growth})
    volatility = getVolatility(cur_ticker)
    volatility_data.append({"name": cur_ticker, "volatility": volatility})

    # Store combined stock details
    stock_details = {
        "name": cur_ticker,
        "beta": beta,
        "growth": growth,
        "volatility": volatility
    }
    stock_data.append(stock_details)

# Sort the data by each criterion
sorted_betas = sorted(beta_data, key=lambda d: d['beta'], reverse=True)
sorted_growth = sorted(growth_data, key=lambda d: d['growth'], reverse=True)
sorted_volatility = sorted(volatility_data, key=lambda d: d['volatility'], reverse=True)

# Initialize a dictionary to store average index values
average_index_dict = {}

# Iterate through each ticker in the stock data
for stock in stock_data:
    ticker_name = stock['name']
    
    # Find the index positions in each sorted list
    beta_index = next(i for i, d in enumerate(sorted_betas) if d['name'] == ticker_name)
    growth_index = next(i for i, d in enumerate(sorted_growth) if d['name'] == ticker_name)
    volatility_index = next(i for i, d in enumerate(sorted_volatility) if d['name'] == ticker_name)
    
    # Calculate the average index position
    average_index = (beta_index + growth_index + volatility_index) / 3
    
    # Store it in the dictionary
    average_index_dict[ticker_name] = average_index

# Sort the average_index_dict by average index values in ascending order
sorted_average_index_dict = {
    k: v for k, v in sorted(average_index_dict.items(), key=lambda item: item[1])
}

# Print the sorted dictionary
print("Sorted Average Index Dictionary:")
print(sorted_average_index_dict)


Sorted Average Index Dictionary:
{'SHOP.TO': 0.0, 'AMZN': 3.3333333333333335, 'QCOM': 3.6666666666666665, 'CAT': 6.666666666666667, 'USB': 7.666666666666667, 'C': 8.0, 'AXP': 8.666666666666666, 'LLY': 9.333333333333334, 'TXN': 10.333333333333334, 'PYPL': 11.333333333333334, 'AAPL': 11.666666666666666, 'BB.TO': 12.666666666666666, 'BAC': 13.333333333333334, 'BK': 14.0, 'ACN': 14.333333333333334, 'BLK': 15.333333333333334, 'AIG': 16.333333333333332, 'BA': 16.666666666666668, 'UPS': 19.333333333333332, 'PM': 20.666666666666668, 'UNP': 21.0, 'MO': 21.333333333333332, 'UNH': 21.333333333333332, 'RY.TO': 23.0, 'BIIB': 23.333333333333332, 'ABT': 24.0, 'BMY': 24.333333333333332, 'PFE': 26.0, 'TD.TO': 26.333333333333332, 'ABBV': 26.666666666666668, 'MRK': 27.666666666666668, 'CL': 28.0, 'LMT': 28.0, 'PG': 29.0, 'KO': 30.333333333333332, 'T.TO': 30.666666666666668, 'PEP': 31.666666666666668}
